In [3]:
from google.colab import files

uploaded = files.upload()

Saving 31952289.zip to 31952289.zip


In [4]:
import zipfile
import os

zip_path = "/content/31952289.zip"
extract_path = "/content/EVChargeIQ_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


# EVChargeIQ — Feature Engineering

## Public EV Charging Transactions & Infrastructure

### Objective

This notebook transforms the audited station and transaction data into analysis-ready features for the EVChargeIQ project.

The feature-engineering process focuses on:

- Temporal charging behavior
- Charging session characteristics
- Energy consumption
- Charging economics
- AC/DC equipment classification
- Station infrastructure
- Station-level utilization metrics

### Design Principles

1. Preserve original source fields.
2. Create derived features rather than overwriting source data.
3. Keep raw data unchanged.
4. Use Parquet for large processed datasets.
5. Validate features on a sample before processing the full transaction dataset.
6. Clearly document every derived metric.

### Input Data

- `stations_public.parquet`
- `orders_2025-01_public.parquet`
- `orders_2025-07_public.parquet`

### Output

Analysis-ready transaction and station datasets will be stored under:

`data/processed/`

In [62]:
import os
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
BASE_PATH = "/content/EVChargeIQ_data"

stations_path = f"{BASE_PATH}/stations_public.parquet"
jan_path = f"{BASE_PATH}/orders_2025-01_public.parquet"
jul_path = f"{BASE_PATH}/orders_2025-07_public.parquet"

print("Dataset paths configured.")

Dataset paths configured.


In [7]:
required_files = [
    stations_path,
    jan_path,
    jul_path
]

for file_path in required_files:
    print(
        os.path.basename(file_path),
        "→",
        "FOUND" if os.path.exists(file_path) else "NOT FOUND"
    )

stations_public.parquet → FOUND
orders_2025-01_public.parquet → FOUND
orders_2025-07_public.parquet → FOUND


## 1. Feature Dictionary

The following derived features will be created from the transaction and station datasets.

### Temporal Features

| Feature | Definition |
|---|---|
| `charge_date` | Calendar date extracted from charging start timestamp |
| `charge_hour` | Hour of charging start |
| `day_of_week` | Day name derived from charging start |
| `day_of_week_num` | Numeric weekday representation |
| `is_weekend` | Whether charging started on Saturday or Sunday |
| `month` | Month of charging start |
| `time_period` | Business-friendly time-of-day category |

### Charging Session Features

| Feature | Definition |
|---|---|
| `avg_charging_power_kw` | Energy consumed divided by session duration |
| `energy_per_minute_kwh` | Energy consumed per charging minute |

### Economic Features

| Feature | Definition |
|---|---|
| `electricity_cost_per_kwh` | Electricity fee divided by energy consumed |
| `service_fee_share` | Service fee divided by total fee |

### Equipment Features

| Feature | Definition |
|---|---|
| `equipment_type` | English standardized label for AC/DC equipment |

### Station Features

Station infrastructure fields will be joined using `station_id`.

These include:

- `construction_site`
- `station_total_power_kw`
- `piles_num`
- `dc_piles_num`
- `ac_piles_num`
- `charging_gun_num`
- `dc_charging_gun_num`
- `ac_charging_gun_num`

### Important Treatment Rules

- Original source columns will not be overwritten.
- Zero-energy transactions will be retained.
- Ratios with a zero-energy denominator will be assigned `NaN` rather than producing infinite values.
- The derived average charging power will not be used as a cleaning rule.
- Chinese source categories will be preserved while standardized English labels are added as new columns.

In [8]:
SAMPLE_SIZE = 10_000

jan_sample = pd.read_parquet(jan_path).head(SAMPLE_SIZE)
jul_sample = pd.read_parquet(jul_path).head(SAMPLE_SIZE)

print("January sample:", jan_sample.shape)
print("July sample:", jul_sample.shape)

January sample: (10000, 12)
July sample: (10000, 12)


In [9]:
sample_transactions = pd.concat(
    [jan_sample, jul_sample],
    ignore_index=True
)

print("Combined sample shape:", sample_transactions.shape)

Combined sample shape: (20000, 12)


## 2. Temporal Feature Engineering

Charging demand is strongly time-dependent.

The charging start timestamp is used to derive calendar and time-of-day features.

These features will support later analysis of:

- Hourly charging demand
- Weekday vs weekend behavior
- Daily charging patterns
- Peak charging periods
- Seasonal comparison between January and July

In [10]:
  # Create temporal features from charge_start_time

sample_transactions["charge_date"] = (
    sample_transactions["charge_start_time"].dt.date
)

sample_transactions["charge_hour"] = (
    sample_transactions["charge_start_time"].dt.hour
)

sample_transactions["day_of_week"] = (
    sample_transactions["charge_start_time"].dt.day_name()
)

sample_transactions["day_of_week_num"] = (
    sample_transactions["charge_start_time"].dt.dayofweek
)

sample_transactions["is_weekend"] = (
    sample_transactions["day_of_week_num"] >= 5
)

sample_transactions["month"] = (
    sample_transactions["charge_start_time"].dt.month
)

print("Temporal features created.")

Temporal features created.


In [11]:
def classify_time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"


sample_transactions["time_period"] = (
    sample_transactions["charge_hour"]
    .apply(classify_time_period)
)

print("Time-period feature created.")

Time-period feature created.


In [12]:
temporal_columns = [
    "charge_start_time",
    "charge_date",
    "charge_hour",
    "day_of_week",
    "day_of_week_num",
    "is_weekend",
    "month",
    "time_period"
]

sample_transactions[temporal_columns].head(10)

,charge_start_time,charge_date,charge_hour,day_of_week,day_of_week_num,is_weekend,month,time_period
0,2025-01-11 20:35:30,2025-01-11,20,Saturday,5,True,1,Evening
1,2025-01-05 19:26:24,2025-01-05,19,Sunday,6,True,1,Evening
2,2025-01-26 14:50:24,2025-01-26,14,Sunday,6,True,1,Afternoon
3,2025-01-29 23:16:55,2025-01-29,23,Wednesday,2,False,1,Night
4,2025-01-18 19:53:45,2025-01-18,19,Saturday,5,True,1,Evening
5,2025-01-15 15:17:53,2025-01-15,15,Wednesday,2,False,1,Afternoon
6,2025-01-02 05:51:42,2025-01-02,5,Thursday,3,False,1,Morning
7,2025-01-21 19:36:15,2025-01-21,19,Tuesday,1,False,1,Evening
8,2025-01-25 17:12:55,2025-01-25,17,Saturday,5,True,1,Evening
9,2025-01-26 18:54:20,2025-01-26,18,Sunday,6,True,1,Evening


In [13]:
print("Hour range:")
print(
    sample_transactions["charge_hour"].min(),
    "to",
    sample_transactions["charge_hour"].max()
)

print("\nDay-of-week range:")
print(
    sample_transactions["day_of_week_num"].min(),
    "to",
    sample_transactions["day_of_week_num"].max()
)

print("\nWeekend values:")
print(
    sample_transactions["is_weekend"].value_counts()
)

print("\nTime periods:")
print(
    sample_transactions["time_period"].value_counts()
)

print("\nMonths:")
print(
    sample_transactions["month"].value_counts().sort_index()
)

Hour range:
0 to 23

Day-of-week range:
0 to 6

Weekend values:
is_weekend
False    17641
True      2359
Name: count, dtype: int64

Time periods:
time_period
Afternoon    5994
Night        5775
Evening      4906
Morning      3325
Name: count, dtype: int64

Months:
month
1    10000
6      414
7     9586
Name: count, dtype: int64


In [14]:
weekend_check = (
    sample_transactions["day_of_week_num"] >= 5
)

print(
    "Weekend classification errors:",
    (
        sample_transactions["is_weekend"] != weekend_check
    ).sum()
)

Weekend classification errors: 0


### Temporal Feature Summary

The following features have been created:

- `charge_date`
- `charge_hour`
- `day_of_week`
- `day_of_week_num`
- `is_weekend`
- `month`
- `time_period`

These features will later be used to analyze charging demand patterns across hours, weekdays, weekends, and the two observed months.

## 3. Charging Session Feature Engineering

The transaction data provides session duration and total energy consumed.

Two derived features are created:

- `avg_charging_power_kw`: estimated average power delivered during the session.
- `energy_per_minute_kwh`: energy consumed per charging minute.

These are session-level analytical metrics.

Zero-energy transactions are retained. Where a derived ratio has a zero-energy denominator, the result will be represented as `NaN` rather than infinity.

In [15]:
# Average charging power
sample_transactions["avg_charging_power_kw"] = (
    sample_transactions["total_elec_kwh"] /
    (sample_transactions["duration_min"] / 60)
)

# Energy consumed per minute
sample_transactions["energy_per_minute_kwh"] = (
    sample_transactions["total_elec_kwh"] /
    sample_transactions["duration_min"]
)

print("Charging session features created.")

Charging session features created.


In [16]:
charging_feature_columns = [
    "duration_min",
    "total_elec_kwh",
    "avg_charging_power_kw",
    "energy_per_minute_kwh"
]

sample_transactions[charging_feature_columns].head(10)

,duration_min,total_elec_kwh,avg_charging_power_kw,energy_per_minute_kwh
0,58.850000,59.30,60.458794,1.007647
1,63.383333,88.58,83.851696,1.397528
2,33.983333,46.74,82.522805,1.375380
3,26.116667,51.89,119.211232,1.986854
4,53.333333,67.00,75.375000,1.256250
5,19.766667,20.98,63.682968,1.061383
6,75.500000,77.91,61.915232,1.031921
7,64.383333,103.27,96.239192,1.603987
8,11.933333,40.82,205.240223,3.420670
9,19.533333,61.65,189.368601,3.156143


In [17]:
sample_transactions[
    ["avg_charging_power_kw", "energy_per_minute_kwh"]
].describe()

,avg_charging_power_kw,energy_per_minute_kwh
count,20000.000000,20000.000000
mean,50.937345,0.848956
std,45.638941,0.760649
min,0.000000,0.000000
25%,19.451361,0.324189
50%,39.141225,0.652354
75%,68.345952,1.139099
max,439.240223,7.320670


In [18]:
print(
    "Infinite average-power values:",
    np.isinf(
        sample_transactions["avg_charging_power_kw"]
    ).sum()
)

print(
    "Infinite energy-per-minute values:",
    np.isinf(
        sample_transactions["energy_per_minute_kwh"]
    ).sum()
)

print(
    "Missing average-power values:",
    sample_transactions["avg_charging_power_kw"].isna().sum()
)

print(
    "Missing energy-per-minute values:",
    sample_transactions["energy_per_minute_kwh"].isna().sum()
)

Infinite average-power values: 0
Infinite energy-per-minute values: 0
Missing average-power values: 0
Missing energy-per-minute values: 0


## 4. Equipment Classification

The source dataset uses Chinese equipment classifications.

The original `equipment_classification` column is preserved.

A new `equipment_type` column provides standardized English labels for analysis.

In [53]:
equipment_mapping = {
    "直流设备": "DC Equipment",
    "交流设备": "AC Equipment"
}

sample_transactions["equipment_type"] = (
    sample_transactions["equipment_classification"]
    .map(equipment_mapping)
)

print(
    sample_transactions[
        ["equipment_classification", "equipment_type"]
    ].drop_duplicates()
)

     equipment_classification equipment_type
0                        直流设备   DC Equipment
7066                     交流设备   AC Equipment


In [20]:
print("Unmapped equipment values:")

print(
    sample_transactions.loc[
        sample_transactions["equipment_type"].isna(),
        "equipment_classification"
    ].unique()
)

Unmapped equipment values:
[]


In [21]:
print(
    sample_transactions["equipment_type"]
    .value_counts()
)

equipment_type
DC Equipment    16160
AC Equipment     3840
Name: count, dtype: int64


## 5. Charging Economics Features

The transaction dataset contains electricity fees, service fees, and total fees.

Two derived metrics are created:

- `electricity_cost_per_kwh`
- `service_fee_share`

Zero-energy transactions cannot produce a meaningful electricity cost per kWh, so their value is represented as `NaN`.

In [22]:
# Initialize with NaN
sample_transactions["electricity_cost_per_kwh"] = np.nan

# Calculate only where energy consumption is greater than zero
energy_mask = sample_transactions["total_elec_kwh"] > 0

sample_transactions.loc[
    energy_mask,
    "electricity_cost_per_kwh"
] = (
    sample_transactions.loc[
        energy_mask,
        "total_elec_fee"
    ]
    /
    sample_transactions.loc[
        energy_mask,
        "total_elec_kwh"
    ]
)

# Service fee share
sample_transactions["service_fee_share"] = np.nan

fee_mask = sample_transactions["total_fee"] > 0

sample_transactions.loc[
    fee_mask,
    "service_fee_share"
] = (
    sample_transactions.loc[
        fee_mask,
        "total_service_fee"
    ]
    /
    sample_transactions.loc[
        fee_mask,
        "total_fee"
    ]
)

print("Economic features created.")

Economic features created.


In [23]:
economic_columns = [
    "total_elec_kwh",
    "total_elec_fee",
    "total_service_fee",
    "total_fee",
    "electricity_cost_per_kwh",
    "service_fee_share"
]

sample_transactions[economic_columns].describe()

,total_elec_kwh,total_elec_fee,total_service_fee,total_fee,electricity_cost_per_kwh,service_fee_share
count,20000.000000,20000.000000,20000.000000,20000.000000,19997.000000,19991.000000
mean,34.482081,29.980053,14.726939,44.706992,0.826021,0.347511
std,24.188525,26.395530,13.817289,35.140831,0.289231,0.161922
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,16.530000,11.340000,5.240000,18.760000,0.622664,0.228534
50%,28.950500,20.560000,10.560000,33.235000,0.838605,0.338114
75%,49.390000,41.570000,20.370000,63.975000,1.018537,0.473576
max,441.521000,251.750000,144.030000,318.290000,2.000000,1.000000


In [24]:
print(
    "Invalid electricity cost values:",
    (
        sample_transactions["electricity_cost_per_kwh"] < 0
    ).sum()
)

print(
    "Invalid service fee shares:",
    (
        (sample_transactions["service_fee_share"] < 0) |
        (sample_transactions["service_fee_share"] > 1)
    ).sum()
)

Invalid electricity cost values: 0
Invalid service fee shares: 0


In [25]:
stations = pd.read_parquet(stations_path)

print("Stations shape:", stations.shape)
print("Unique station IDs:", stations["station_id"].nunique())

Stations shape: (8553, 10)
Unique station IDs: 8553


In [26]:
station_features = [
    "station_id",
    "geocoding",
    "construction_site",
    "station_total_power_kw",
    "piles_num",
    "dc_piles_num",
    "ac_piles_num",
    "charging_gun_num",
    "dc_charging_gun_num",
    "ac_charging_gun_num"
]

station_lookup = stations[station_features].copy()

print("Station lookup shape:", station_lookup.shape)

Station lookup shape: (8553, 10)


In [27]:
print(
    "Duplicate station IDs in lookup:",
    station_lookup["station_id"].duplicated().sum()
)

Duplicate station IDs in lookup: 0


In [28]:
sample_with_station = sample_transactions.merge(
    station_lookup,
    on="station_id",
    how="left",
    validate="many_to_one"
)

print("Original transaction sample:", sample_transactions.shape)
print("After station merge:", sample_with_station.shape)

Original transaction sample: (20000, 24)
After station merge: (20000, 33)


In [29]:
station_join_missing = (
    sample_with_station["station_total_power_kw"]
    .isna()
    .sum()
)

print(
    "Transactions without matching station infrastructure:",
    station_join_missing
)

print(
    "Station join coverage:",
    f"{(1 - station_join_missing / len(sample_with_station)) * 100:.2f}%"
)

Transactions without matching station infrastructure: 0
Station join coverage: 100.00%


In [30]:
station_feature_columns = [
    "station_id",
    "construction_site",
    "station_total_power_kw",
    "piles_num",
    "dc_piles_num",
    "ac_piles_num",
    "charging_gun_num",
    "dc_charging_gun_num",
    "ac_charging_gun_num"
]

sample_with_station[
    station_feature_columns
].head(10)

,station_id,construction_site,station_total_power_kw,piles_num,dc_piles_num,ac_piles_num,charging_gun_num,dc_charging_gun_num,ac_charging_gun_num
0,ST_007597,大型建筑配建停车场,960.0,8.0,8.0,0.0,8.0,8.0,0.0
1,ST_007597,大型建筑配建停车场,960.0,8.0,8.0,0.0,8.0,8.0,0.0
2,ST_007597,大型建筑配建停车场,960.0,8.0,8.0,0.0,8.0,8.0,0.0
3,ST_007597,大型建筑配建停车场,960.0,8.0,8.0,0.0,8.0,8.0,0.0
4,ST_007597,大型建筑配建停车场,960.0,8.0,8.0,0.0,8.0,8.0,0.0
5,ST_007597,大型建筑配建停车场,960.0,8.0,8.0,0.0,8.0,8.0,0.0
6,ST_007597,大型建筑配建停车场,960.0,8.0,8.0,0.0,8.0,8.0,0.0
7,ST_007597,大型建筑配建停车场,960.0,8.0,8.0,0.0,8.0,8.0,0.0
8,ST_007597,大型建筑配建停车场,960.0,8.0,8.0,0.0,8.0,8.0,0.0
9,ST_007597,大型建筑配建停车场,960.0,8.0,8.0,0.0,8.0,8.0,0.0


## 6. Station Infrastructure Features

Station infrastructure characteristics are joined to transactions using `station_id`.

Additional station-level ratios will be created to describe infrastructure composition.

These ratios are descriptive features and do not alter the original station counts.

In [31]:
# DC pile share
sample_with_station["dc_pile_share"] = np.where(
    sample_with_station["piles_num"] > 0,
    sample_with_station["dc_piles_num"] /
    sample_with_station["piles_num"],
    np.nan
)

# DC charging-gun share
sample_with_station["dc_gun_share"] = np.where(
    sample_with_station["charging_gun_num"] > 0,
    sample_with_station["dc_charging_gun_num"] /
    sample_with_station["charging_gun_num"],
    np.nan
)

# Station power per pile
sample_with_station["power_per_pile_kw"] = np.where(
    sample_with_station["piles_num"] > 0,
    sample_with_station["station_total_power_kw"] /
    sample_with_station["piles_num"],
    np.nan
)

print("Station infrastructure features created.")

Station infrastructure features created.


In [32]:
infrastructure_features = [
    "dc_pile_share",
    "dc_gun_share",
    "power_per_pile_kw"
]

print(
    sample_with_station[
        infrastructure_features
    ].describe()
)

       dc_pile_share  dc_gun_share  power_per_pile_kw
count   20000.000000  20000.000000       20000.000000
mean        0.789722      0.795651         139.976928
std         0.393986      0.389508         163.998349
min         0.000000      0.000000           4.375000
25%         1.000000      1.000000          60.000000
50%         1.000000      1.000000          90.000000
75%         1.000000      1.000000         137.142857
max         1.000000      1.000000        1920.000000


In [33]:
print(
    "Invalid DC pile shares:",
    (
        (sample_with_station["dc_pile_share"] < 0) |
        (sample_with_station["dc_pile_share"] > 1)
    ).sum()
)

print(
    "Invalid DC gun shares:",
    (
        (sample_with_station["dc_gun_share"] < 0) |
        (sample_with_station["dc_gun_share"] > 1)
    ).sum()
)

Invalid DC pile shares: 0
Invalid DC gun shares: 0


## 7. Final Feature Inventory

The engineered transaction dataset combines:

1. Original transaction attributes
2. Temporal features
3. Charging-session features
4. Economic features
5. Standardized equipment classification
6. Station infrastructure attributes
7. Station infrastructure ratios

The original source columns are retained throughout the transformation process.

In [34]:
print("Final engineered sample shape:")
print(sample_with_station.shape)

print("\nFinal feature list:")
for i, column in enumerate(sample_with_station.columns, start=1):
    print(f"{i:02d}. {column}")

Final engineered sample shape:
(20000, 36)

Final feature list:
01. transaction_date
02. equipment_classification
03. rated_gun_power_kw
04. station_id
05. pile_id
06. duration_min
07. charge_start_time
08. charge_end_time
09. total_elec_kwh
10. total_elec_fee
11. total_service_fee
12. total_fee
13. charge_date
14. charge_hour
15. day_of_week
16. day_of_week_num
17. is_weekend
18. month
19. time_period
20. avg_charging_power_kw
21. energy_per_minute_kwh
22. equipment_type
23. electricity_cost_per_kwh
24. service_fee_share
25. geocoding
26. construction_site
27. station_total_power_kw
28. piles_num
29. dc_piles_num
30. ac_piles_num
31. charging_gun_num
32. dc_charging_gun_num
33. ac_charging_gun_num
34. dc_pile_share
35. dc_gun_share
36. power_per_pile_kw


In [35]:
print(
    "Duplicate rows after feature engineering:",
    sample_with_station.duplicated().sum()
)

Duplicate rows after feature engineering: 0


In [36]:
final_missing = (
    sample_with_station.isnull().sum()
    .sort_values(ascending=False)
)

final_missing[final_missing > 0]

,0
service_fee_share,9
electricity_cost_per_kwh,3


In [51]:
def engineer_transaction_features(df, station_lookup):
    """
    Transform raw EV charging transactions into an analysis-ready dataset.
    """

    df = df.copy()

    # -----------------------------
    # Temporal features
    # -----------------------------
    df["charge_date"] = df["charge_start_time"].dt.date
    df["charge_hour"] = df["charge_start_time"].dt.hour
    df["day_of_week"] = df["charge_start_time"].dt.day_name()
    df["day_of_week_num"] = df["charge_start_time"].dt.dayofweek
    df["is_weekend"] = df["day_of_week_num"] >= 5
    df["month"] = df["charge_start_time"].dt.month

    def classify_time_period(hour):
        if 5 <= hour < 12:
            return "Morning"
        elif 12 <= hour < 17:
            return "Afternoon"
        elif 17 <= hour < 21:
            return "Evening"
        else:
            return "Night"

    df["time_period"] = df["charge_hour"].apply(
        classify_time_period
    )

    # -----------------------------
    # Charging session features
    # -----------------------------
    df["avg_charging_power_kw"] = np.where(
        df["duration_min"] > 0,
        df["total_elec_kwh"] /
        (df["duration_min"] / 60),
        np.nan
    )

    df["energy_per_minute_kwh"] = np.where(
        df["duration_min"] > 0,
        df["total_elec_kwh"] /
        df["duration_min"],
        np.nan
    )

    # -----------------------------
    # Equipment standardization
    # -----------------------------
    equipment_mapping = {
        "直流设备": "DC Equipment",
        "交流设备": "AC Equipment"
    }

    df["equipment_type"] = (
        df["equipment_classification"]
        .map(equipment_mapping)
        .fillna("Unknown / Missing")
    )

    # -----------------------------
    # Economic features
    # -----------------------------
    df["electricity_cost_per_kwh"] = np.where(
        df["total_elec_kwh"] > 0,
        df["total_elec_fee"] /
        df["total_elec_kwh"],
        np.nan
    )

    df["service_fee_share"] = np.where(
        df["total_fee"] > 0,
        df["total_service_fee"] /
        df["total_fee"],
        np.nan
    )

    # -----------------------------
    # Station infrastructure join
    # -----------------------------
    df = df.merge(
        station_lookup,
        on="station_id",
        how="left",
        validate="many_to_one"
    )

    # -----------------------------
    # Infrastructure features
    # -----------------------------
    df["dc_pile_share"] = np.where(
        df["piles_num"] > 0,
        df["dc_piles_num"] /
        df["piles_num"],
        np.nan
    )

    df["dc_gun_share"] = np.where(
        df["charging_gun_num"] > 0,
        df["dc_charging_gun_num"] /
        df["charging_gun_num"],
        np.nan
    )

    df["power_per_pile_kw"] = np.where(
        df["piles_num"] > 0,
        df["station_total_power_kw"] /
        df["piles_num"],
        np.nan
    )

    return df


print("Feature-engineering function updated successfully.")

Feature-engineering function updated successfully.


In [54]:
engineered_sample = engineer_transaction_features(
    pd.concat(
        [
            pd.read_parquet(jan_path).head(10_000),
            pd.read_parquet(jul_path).head(10_000)
        ],
        ignore_index=True
    ),
    station_lookup
)

print("Engineered sample shape:", engineered_sample.shape)

Engineered sample shape: (20000, 36)


In [55]:
print(
    "Unknown / Missing equipment:",
    (
        engineered_sample["equipment_type"]
        == "Unknown / Missing"
    ).sum()
)

print(
    "Remaining unmapped equipment:",
    engineered_sample["equipment_type"].isna().sum()
)

Unknown / Missing equipment: 0
Remaining unmapped equipment: 0


In [39]:
print("Rows:", len(engineered_sample))

print(
    "Duplicate rows:",
    engineered_sample.duplicated().sum()
)

print(
    "Unmapped equipment types:",
    engineered_sample["equipment_type"].isna().sum()
)

print(
    "Missing station matches:",
    engineered_sample["station_total_power_kw"].isna().sum()
)

print(
    "Infinite average-power values:",
    np.isinf(
        engineered_sample["avg_charging_power_kw"]
    ).sum()
)

Rows: 20000
Duplicate rows: 0
Unmapped equipment types: 0
Missing station matches: 0
Infinite average-power values: 0


In [56]:
jan_test = pd.read_parquet(
    jan_path,
    columns=[
        "transaction_date",
        "equipment_classification",
        "rated_gun_power_kw",
        "station_id",
        "pile_id",
        "duration_min",
        "charge_start_time",
        "charge_end_time",
        "total_elec_kwh",
        "total_elec_fee",
        "total_service_fee",
        "total_fee"
    ]
)

jan_test_engineered = engineer_transaction_features(
    jan_test,
    station_lookup
)

print("January rows:", len(jan_test_engineered))

print(
    "\nEquipment types:"
)

print(
    jan_test_engineered["equipment_type"]
    .value_counts(dropna=False)
)

January rows: 2137234

Equipment types:
equipment_type
DC Equipment         1938830
AC Equipment          197225
Unknown / Missing       1179
Name: count, dtype: int64


## 8. Full Dataset Feature Engineering

The validated feature-engineering function is now applied to the complete January and July transaction datasets.

Because the transaction data contains more than 8.5 million records, the files are processed in Parquet row groups rather than loading the complete datasets into memory at once.

The processed outputs are stored as Parquet files.

In [40]:
PROCESSED_PATH = "/content/EVChargeIQ_processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Processed-data directory:")
print(PROCESSED_PATH)

Processed-data directory:
/content/EVChargeIQ_processed


In [42]:
# Inspect Parquet file structure

jan_file = pq.ParquetFile(jan_path)
jul_file = pq.ParquetFile(jul_path)

print("January row groups:", jan_file.metadata.num_row_groups)
print("July row groups:", jul_file.metadata.num_row_groups)

print("\nJanuary rows:", jan_file.metadata.num_rows)
print("July rows:", jul_file.metadata.num_rows)

January row groups: 3
July row groups: 7

January rows: 2137234
July rows: 6407461


In [43]:
def process_parquet_to_parquet(
    input_path,
    output_path,
    station_lookup,
    batch_size=100_000
):
    """
    Process a large Parquet transaction file in batches
    and save the engineered result as Parquet.
    """

    parquet_file = pq.ParquetFile(input_path)

    processed_batches = []
    total_rows = 0

    for batch in parquet_file.iter_batches(
        batch_size=batch_size
    ):
        batch_df = batch.to_pandas()

        engineered_batch = engineer_transaction_features(
            batch_df,
            station_lookup
        )

        processed_batches.append(engineered_batch)
        total_rows += len(engineered_batch)

        print(
            f"Processed rows: {total_rows:,}",
            end="\r"
        )

    processed_df = pd.concat(
        processed_batches,
        ignore_index=True
    )

    processed_df.to_parquet(
        output_path,
        index=False
    )

    print(
        f"\nSaved {total_rows:,} rows to:"
    )
    print(output_path)

    return processed_df

In [44]:
jan_processed_path = (
    f"{PROCESSED_PATH}/"
    "transactions_2025-01_processed.parquet"
)

jan_processed = process_parquet_to_parquet(
    input_path=jan_path,
    output_path=jan_processed_path,
    station_lookup=station_lookup,
    batch_size=100_000
)


Saved 2,137,234 rows to:
/content/EVChargeIQ_processed/transactions_2025-01_processed.parquet


In [57]:
# Reprocess January using the corrected feature-engineering function

jan_processed_path = (
    f"{PROCESSED_PATH}/"
    "transactions_2025-01_processed.parquet"
)

jan_processed = process_parquet_to_parquet(
    input_path=jan_path,
    output_path=jan_processed_path,
    station_lookup=station_lookup,
    batch_size=100_000
)


Saved 2,137,234 rows to:
/content/EVChargeIQ_processed/transactions_2025-01_processed.parquet


In [45]:
jan_output_file = pq.ParquetFile(jan_processed_path)

print("January processed file verified.")
print("Rows:", jan_output_file.metadata.num_rows)
print("Columns:", jan_output_file.metadata.num_columns)
print("Row groups:", jan_output_file.metadata.num_row_groups)

January processed file verified.
Rows: 2137234
Columns: 36
Row groups: 3


In [46]:
jan_processed_check = pd.read_parquet(
    jan_processed_path
).head(5)

print("Processed January columns:")
print(jan_processed_check.columns.tolist())

Processed January columns:
['transaction_date', 'equipment_classification', 'rated_gun_power_kw', 'station_id', 'pile_id', 'duration_min', 'charge_start_time', 'charge_end_time', 'total_elec_kwh', 'total_elec_fee', 'total_service_fee', 'total_fee', 'charge_date', 'charge_hour', 'day_of_week', 'day_of_week_num', 'is_weekend', 'month', 'time_period', 'avg_charging_power_kw', 'energy_per_minute_kwh', 'equipment_type', 'electricity_cost_per_kwh', 'service_fee_share', 'geocoding', 'construction_site', 'station_total_power_kw', 'piles_num', 'dc_piles_num', 'ac_piles_num', 'charging_gun_num', 'dc_charging_gun_num', 'ac_charging_gun_num', 'dc_pile_share', 'dc_gun_share', 'power_per_pile_kw']


In [47]:
jan_validation = pd.read_parquet(
    jan_processed_path,
    columns=[
        "station_id",
        "equipment_type",
        "charge_hour",
        "is_weekend",
        "avg_charging_power_kw",
        "electricity_cost_per_kwh",
        "service_fee_share",
        "station_total_power_kw"
    ]
)

print("Rows loaded for validation:", len(jan_validation))

print(
    "Unmapped equipment:",
    jan_validation["equipment_type"].isna().sum()
)

print(
    "Missing station capacity:",
    jan_validation["station_total_power_kw"].isna().sum()
)

print(
    "Infinite average power:",
    np.isinf(
        jan_validation["avg_charging_power_kw"]
    ).sum()
)

Rows loaded for validation: 2137234
Unmapped equipment: 1179
Missing station capacity: 0
Infinite average power: 0


In [48]:
# Investigate equipment classifications that were not mapped

unmapped_equipment = (
    jan_processed_path
)

jan_equipment_check = pd.read_parquet(
    unmapped_equipment,
    columns=["equipment_classification", "equipment_type"]
)

unmapped_values = (
    jan_equipment_check.loc[
        jan_equipment_check["equipment_type"].isna(),
        "equipment_classification"
    ]
    .value_counts(dropna=False)
)

print("Unmapped equipment classifications:")
print(unmapped_values)

print(
    "\nTotal unmapped records:",
    unmapped_values.sum()
)

Unmapped equipment classifications:
equipment_classification
None    1179
Name: count, dtype: int64

Total unmapped records: 1179


In [49]:
print(
    "\nAll unique equipment classifications in January:"
)

print(
    jan_equipment_check[
        "equipment_classification"
    ].value_counts(dropna=False)
)


All unique equipment classifications in January:
equipment_classification
直流设备    1938830
交流设备     197225
None       1179
Name: count, dtype: int64


In [58]:
jan_output_file = pq.ParquetFile(jan_processed_path)

print("Rows:", jan_output_file.metadata.num_rows)
print("Columns:", jan_output_file.metadata.num_columns)

Rows: 2137234
Columns: 36


In [59]:
jan_equipment_final = pd.read_parquet(
    jan_processed_path,
    columns=["equipment_type"]
)

print(
    jan_equipment_final["equipment_type"]
    .value_counts(dropna=False)
)

print(
    "\nMissing equipment_type:",
    jan_equipment_final["equipment_type"].isna().sum()
)

equipment_type
DC Equipment         1938830
AC Equipment          197225
Unknown / Missing       1179
Name: count, dtype: int64

Missing equipment_type: 0


In [60]:
from pyarrow import parquet as pq

In [61]:
def process_parquet_streaming(
    input_path,
    output_path,
    station_lookup,
    batch_size=100_000
):
    """
    Process a large Parquet transaction file in batches
    and write each engineered batch directly to a Parquet file.
    """

    input_file = pq.ParquetFile(input_path)

    writer = None
    total_rows = 0

    try:
        for batch in input_file.iter_batches(
            batch_size=batch_size
        ):
            batch_df = batch.to_pandas()

            engineered_batch = engineer_transaction_features(
                batch_df,
                station_lookup
            )

            table = pa.Table.from_pandas(
                engineered_batch,
                preserve_index=False
            )

            if writer is None:
                writer = pq.ParquetWriter(
                    output_path,
                    table.schema
                )

            writer.write_table(table)

            total_rows += len(engineered_batch)

            print(
                f"Processed rows: {total_rows:,}",
                end="\r"
            )

    finally:
        if writer is not None:
            writer.close()

    print(
        f"\nSaved {total_rows:,} rows to:"
    )
    print(output_path)

    return total_rows

In [63]:
jul_processed_path = (
    f"{PROCESSED_PATH}/"
    "transactions_2025-07_processed.parquet"
)

jul_processed_rows = process_parquet_streaming(
    input_path=jul_path,
    output_path=jul_processed_path,
    station_lookup=station_lookup,
    batch_size=100_000
)

Processed rows: 6,407,461
Saved 6,407,461 rows to:
/content/EVChargeIQ_processed/transactions_2025-07_processed.parquet


In [64]:
jul_output_file = pq.ParquetFile(jul_processed_path)

print("July processed file verified.")
print("Rows:", jul_output_file.metadata.num_rows)
print("Columns:", jul_output_file.metadata.num_columns)
print("Row groups:", jul_output_file.metadata.num_row_groups)

July processed file verified.
Rows: 6407461
Columns: 36
Row groups: 65


In [65]:
jul_equipment_final = pd.read_parquet(
    jul_processed_path,
    columns=[
        "equipment_classification",
        "equipment_type"
    ]
)

print(
    jul_equipment_final["equipment_classification"]
    .value_counts(dropna=False)
)

print("\nStandardized equipment types:")
print(
    jul_equipment_final["equipment_type"]
    .value_counts(dropna=False)
)

print(
    "\nMissing equipment_type:",
    jul_equipment_final["equipment_type"].isna().sum()
)

equipment_classification
直流设备    5889298
交流设备     518163
Name: count, dtype: int64

Standardized equipment types:
equipment_type
DC Equipment    5889298
AC Equipment     518163
Name: count, dtype: int64

Missing equipment_type: 0


In [66]:
jul_station_check = pd.read_parquet(
    jul_processed_path,
    columns=[
        "station_id",
        "station_total_power_kw"
    ]
)

print(
    "Rows loaded:",
    len(jul_station_check)
)

print(
    "Missing station matches:",
    jul_station_check["station_total_power_kw"].isna().sum()
)

Rows loaded: 6407461
Missing station matches: 0


In [67]:
jul_feature_check = pd.read_parquet(
    jul_processed_path,
    columns=[
        "avg_charging_power_kw",
        "energy_per_minute_kwh",
        "electricity_cost_per_kwh",
        "service_fee_share",
        "dc_pile_share",
        "dc_gun_share",
        "power_per_pile_kw"
    ]
)

print(
    "Infinite average-power values:",
    np.isinf(
        jul_feature_check["avg_charging_power_kw"]
    ).sum()
)

print(
    "Infinite energy-per-minute values:",
    np.isinf(
        jul_feature_check["energy_per_minute_kwh"]
    ).sum()
)

print(
    "Invalid service-fee shares:",
    (
        (jul_feature_check["service_fee_share"] < 0) |
        (jul_feature_check["service_fee_share"] > 1)
    ).sum()
)

print(
    "Invalid DC pile shares:",
    (
        (jul_feature_check["dc_pile_share"] < 0) |
        (jul_feature_check["dc_pile_share"] > 1)
    ).sum()
)

print(
    "Invalid DC gun shares:",
    (
        (jul_feature_check["dc_gun_share"] < 0) |
        (jul_feature_check["dc_gun_share"] > 1)
    ).sum()
)


Infinite average-power values: 0
Infinite energy-per-minute values: 0
Invalid service-fee shares: 0
Invalid DC pile shares: 0
Invalid DC gun shares: 0
